# Silver Transactions

Convert released Bronze transactions into trusted monitoring inputs.

Input: workspace.sentinel_bronze.transactions

Planned outputs:
- Accepted transactions.
- Quarantined records with validation reasons.

Begin by inspecting schema and amount precision.
No tables are written during this inspection.

In [ ]:
from pyspark.sql import functions as F

# Read the persisted Bronze table, not future staged batches.
bronze_table = "workspace.sentinel_bronze.transactions"
bronze = spark.table(bronze_table)

# Inspect actual types before defining Silver transformations.
bronze.printSchema()

# Labels must remain outside operational data.
assert "is_fraud" not in bronze.columns, (
    "Unexpected fraud labels in Bronze"
)

# Confirm which released batches are available.
display(
    bronze
    .groupBy("replay_batch")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy("replay_batch")
)

# Compare a wider decimal representation with the intended monetary type.
# A successful DECIMAL(18,2) cast alone does not prove that no rounding occurred.
amount_precision = bronze.select(
    F.expr("TRY_CAST(amt AS DECIMAL(38, 18))").alias("wide_amount"),
    F.expr("TRY_CAST(amt AS DECIMAL(18, 2))").alias("money_amount")
)

display(
    amount_precision.agg(
        F.count("*").alias("input_rows"),
        F.count_if(
            F.col("wide_amount").isNull()
            | F.col("money_amount").isNull()
        ).alias("unparseable_or_out_of_range_amounts"),
        F.count_if(
            F.col("wide_amount")
            != F.col("money_amount").cast("decimal(38,18)")
        ).alias("amounts_changed_at_two_decimal_places")
    )
)

### Inspect Bronze rescued data

Check whether Auto Loader populated `_rescued_data` in Bronze and inspect any
rescued JSON keys before deciding whether `_rescued_data` should trigger a
validation reason.

In [ ]:
from pyspark.sql import DataFrame


def summarize_rescued_data(
    df: DataFrame,
) -> tuple[DataFrame, DataFrame]:
    """Return rescued-data row counts and distinct JSON key combinations."""
    rescued_counts = df.agg(
        F.count("*").alias("input_rows"),
        F.count_if(F.col("_rescued_data").isNotNull()).alias(
            "rows_with_non_null_rescued_data"
        ),
        F.count_if(
            F.col("_rescued_data").isNotNull()
            & (F.trim(F.col("_rescued_data")) != "")
        ).alias("rows_with_non_blank_rescued_data")
    )

    rescued_keys = (
        df.filter(
            F.col("_rescued_data").isNotNull()
            & (F.trim(F.col("_rescued_data")) != "")
        )
        .select(
            F.array_sort(
                F.map_keys(
                    F.from_json(F.col("_rescued_data"), "MAP<STRING,STRING>")
                )
            ).alias("rescued_keys")
        )
        .groupBy("rescued_keys")
        .agg(F.count("*").alias("row_count"))
        .orderBy(F.col("row_count").desc())
    )
    return rescued_counts, rescued_keys


rescued_count_summary, rescued_key_summary = summarize_rescued_data(bronze)
display(rescued_count_summary)
display(rescued_key_summary)

### Normalize required fields and convert types

- Normalize blank required identifiers (`trans_num`, `cc_num`, `merchant`) to `NULL`.
- Parse `trans_date_trans_time` as `TIMESTAMP_NTZ` (`event_time`) without assuming a timezone.
- Convert `amt` to `DECIMAL(18,2)` (`amount`) alongside `DECIMAL(38,18)` (`_wide_amount`).
- Preserve raw source values and lineage columns for investigation.

In [ ]:
# Lineage metadata columns are excluded from the business-record signature
# so duplicate classification compares only transaction business attributes.
LINEAGE_COLUMNS = {
    "source_file",
    "source_row_index",
    "replay_batch",
    "_rescued_data",
    "_ingested_file_path",
    "_ingestion_timestamp",
}


def normalize_and_type_transactions(df: DataFrame) -> DataFrame:
    """Normalize required identifiers, cast types, and compute a business signature."""
    business_columns = sorted([
        col_name for col_name in df.columns
        if col_name not in LINEAGE_COLUMNS
    ])

    typed_df = (
        df
        .withColumn("raw_trans_num", F.col("trans_num"))
        .withColumn("raw_cc_num", F.col("cc_num"))
        .withColumn("raw_merchant", F.col("merchant"))
        .withColumn("raw_amt", F.col("amt"))
        .withColumn("raw_trans_date_trans_time", F.col("trans_date_trans_time"))
        .withColumn("trans_num", F.expr("NULLIF(TRIM(trans_num), '')"))
        .withColumn("cc_num", F.expr("NULLIF(TRIM(cc_num), '')"))
        .withColumn("merchant", F.expr("NULLIF(TRIM(merchant), '')"))
        .withColumn(
            "_normalized_event_time_str",
            F.expr("NULLIF(TRIM(trans_date_trans_time), '')")
        )
        .withColumn(
            "event_time",
            F.expr("TRY_CAST(_normalized_event_time_str AS TIMESTAMP_NTZ)")
        )
        .withColumn(
            "_normalized_amt_str",
            F.expr("NULLIF(TRIM(amt), '')")
        )
        .withColumn(
            "_wide_amount",
            F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(38, 18))")
        )
        .withColumn(
            "amount",
            F.expr("TRY_CAST(_normalized_amt_str AS DECIMAL(18, 2))")
        )
    )

    # Build a deterministic JSON signature over all source business columns.
    # Using normalized strings (with an explicit sentinel for NULL) ensures
    # that invalid rows still participate in conflict detection.
    business_signature_struct = F.struct(*[
        F.coalesce(
            F.expr(f"NULLIF(TRIM(`{col_name}`), '')").cast("string"),
            F.lit("__NULL__")
        ).alias(col_name)
        for col_name in business_columns
    ])

    return typed_df.withColumn(
        "_business_signature",
        F.to_json(business_signature_struct)
    )


typed_transactions = normalize_and_type_transactions(bronze)

### Assign field validation reasons

Evaluate every required identifier, event timestamp, and monetary amount check.
Preserve all applicable reasons on each row.

In [ ]:
def assign_field_validation_reasons(typed_df: DataFrame) -> DataFrame:
    """Attach all applicable field-level validation failure reasons to each row."""
    raw_validation_reasons = F.array(
        F.when(
            F.col("trans_num").isNull(),
            F.lit("missing_transaction_id")
        ),
        F.when(
            F.col("cc_num").isNull(),
            F.lit("missing_card_id")
        ),
        F.when(
            F.col("merchant").isNull(),
            F.lit("missing_merchant")
        ),
        F.when(
            F.col("_normalized_event_time_str").isNull(),
            F.lit("missing_event_time")
        ),
        F.when(
            F.col("_normalized_event_time_str").isNotNull()
            & F.col("event_time").isNull(),
            F.lit("unparseable_event_time")
        ),
        F.when(
            F.col("_normalized_amt_str").isNull(),
            F.lit("missing_amount")
        ),
        F.when(
            F.col("_normalized_amt_str").isNotNull()
            & (F.col("_wide_amount").isNull() | F.col("amount").isNull()),
            F.lit("unparseable_or_out_of_range_amount")
        ),
        F.when(
            F.col("_wide_amount").isNotNull()
            & F.col("amount").isNotNull()
            & (
                F.col("_wide_amount")
                != F.col("amount").cast("decimal(38,18)")
            ),
            F.lit("amount_precision_changed")
        ),
        F.when(
            F.col("_wide_amount").isNotNull()
            & (F.col("_wide_amount") <= F.lit(0).cast("decimal(38,18)")),
            F.lit("non_positive_amount")
        )
    )

    return typed_df.withColumn(
        "field_validation_reasons",
        F.filter(raw_validation_reasons, lambda reason: reason.isNotNull())
    )


validated_transactions = assign_field_validation_reasons(typed_transactions)

### Classify duplicate transaction IDs across Bronze

- Check `trans_num` multiplicity across the entire Bronze input before filtering invalid rows.
- Distinguish identical business records (`distinct_business_signatures == 1`) from conflicting records (`distinct_business_signatures > 1`).
- Quarantine all rows involved in a conflicting duplicate ID without selecting an arbitrary winner.
- For valid identical duplicates, retain one deterministic copy as `accepted` and classify extra copies as `duplicate_identical`.

In [ ]:
from pyspark.sql.window import Window


def classify_duplicate_transactions(validated_df: DataFrame) -> DataFrame:
    """Classify duplicate transaction IDs and assign a mutually exclusive disposition."""
    # Summarize occurrence count and distinct business payloads per non-null ID
    # across all input rows, including rows with field validation errors.
    duplicate_summary = (
        validated_df
        .filter(F.col("trans_num").isNotNull())
        .groupBy("trans_num")
        .agg(
            F.count("*").alias("id_occurrence_count"),
            F.countDistinct("_business_signature").alias(
                "distinct_business_signatures"
            )
        )
    )

    # Order occurrences deterministically within each non-null trans_num to
    # retain a single copy only when duplicate records have identical business values.
    id_order_window = Window.partitionBy("trans_num").orderBy(
        F.col("_ingestion_timestamp").asc_nulls_last(),
        F.col("_ingested_file_path").asc_nulls_last(),
        F.expr("TRY_CAST(source_row_index AS BIGINT)").asc_nulls_last(),
        F.col("source_row_index").asc_nulls_last()
    )

    return (
        validated_df
        .join(duplicate_summary, on="trans_num", how="left")
        .withColumn(
            "duplicate_rank",
            F.when(
                F.col("trans_num").isNotNull(),
                F.row_number().over(id_order_window)
            )
        )
        .withColumn(
            "duplicate_status",
            F.when(F.col("trans_num").isNull(), F.lit("missing_id"))
            .when(F.col("id_occurrence_count") == 1, F.lit("unique_id"))
            .when(
                (F.col("id_occurrence_count") > 1)
                & (F.col("distinct_business_signatures") > 1),
                F.lit("conflicting_duplicate")
            )
            .when(
                (F.col("id_occurrence_count") > 1)
                & (F.col("distinct_business_signatures") == 1)
                & (F.col("duplicate_rank") == 1),
                F.lit("identical_duplicate_primary")
            )
            .otherwise(F.lit("identical_duplicate_extra"))
        )
        .withColumn(
            "duplicate_validation_reasons",
            F.when(
                F.col("duplicate_status") == "conflicting_duplicate",
                F.array(F.lit("conflicting_duplicate_transaction_id"))
            )
            .when(
                F.col("duplicate_status") == "identical_duplicate_extra",
                F.array(F.lit("duplicate_identical_transaction_id"))
            )
            .otherwise(F.array().cast("array<string>"))
        )
        .withColumn(
            "validation_reasons",
            F.concat(
                F.col("field_validation_reasons"),
                F.col("duplicate_validation_reasons")
            )
        )
        .withColumn(
            "record_disposition",
            F.when(
                F.size(F.col("validation_reasons")) == 0,
                F.lit("accepted")
            )
            .when(
                (F.col("duplicate_status") == "identical_duplicate_extra")
                & (F.size(F.col("field_validation_reasons")) == 0),
                F.lit("duplicate_identical")
            )
            .otherwise(F.lit("quarantined"))
        )
    )


def classify_silver_transactions(df: DataFrame) -> DataFrame:
    """Run the full Silver typing, validation, and duplicate classification pipeline."""
    typed_df = normalize_and_type_transactions(df)
    validated_df = assign_field_validation_reasons(typed_df)
    return classify_duplicate_transactions(validated_df)


classified_transactions = classify_duplicate_transactions(
    validated_transactions
)

# Separate intermediate DataFrames by disposition (no tables written yet).
accepted_transactions = classified_transactions.filter(
    F.col("record_disposition") == "accepted"
)
quarantined_records = classified_transactions.filter(
    F.col("record_disposition") == "quarantined"
)
duplicate_identical_records = classified_transactions.filter(
    F.col("record_disposition") == "duplicate_identical"
)

### Reconcile rows and inspect validation results

Verify that every Bronze input row is accounted for exactly once and that
accepted transaction IDs are unique.

In [ ]:
def assert_and_summarize_reconciliation(
    input_df: DataFrame,
    classified_df: DataFrame,
) -> dict[str, int]:
    """Verify row-accounting and accepted-key invariants and return summary counts."""
    metrics_row = classified_df.agg(
        F.count("*").alias("classified_rows"),
        F.count_if(F.col("record_disposition").isNull()).alias(
            "unclassified_rows"
        ),
        F.count_if(F.col("record_disposition") == "accepted").alias(
            "accepted_rows"
        ),
        F.count_if(F.col("record_disposition") == "quarantined").alias(
            "quarantined_rows"
        ),
        F.count_if(F.col("record_disposition") == "duplicate_identical").alias(
            "duplicate_identical_rows"
        ),
        F.count_if(
            (F.col("record_disposition") == "accepted")
            & (
                F.col("trans_num").isNull()
                | (F.size(F.col("validation_reasons")) > 0)
            )
        ).alias("invalid_accepted_rows"),
        F.count_if(
            (F.col("record_disposition") != "accepted")
            & (F.size(F.col("validation_reasons")) == 0)
        ).alias("non_accepted_without_reasons"),
        F.count_if(F.col("_rescued_data").isNotNull()).alias(
            "rows_with_rescued_data"
        )
    ).collect()[0]

    input_rows = input_df.count()
    distinct_accepted_ids = (
        classified_df
        .filter(F.col("record_disposition") == "accepted")
        .select("trans_num")
        .distinct()
        .count()
    )

    assert metrics_row["classified_rows"] == input_rows, (
        "Classified row count does not match input count"
    )
    assert metrics_row["unclassified_rows"] == 0, (
        "Every input row must receive a disposition"
    )
    assert (
        metrics_row["accepted_rows"]
        + metrics_row["quarantined_rows"]
        + metrics_row["duplicate_identical_rows"]
        == input_rows
    ), "Disposition counts do not sum to input count"
    assert metrics_row["invalid_accepted_rows"] == 0, (
        "Accepted rows must have a non-null trans_num and no validation reasons"
    )
    assert metrics_row["non_accepted_without_reasons"] == 0, (
        "Every quarantined or duplicate row must record a validation reason"
    )
    assert distinct_accepted_ids == metrics_row["accepted_rows"], (
        "Accepted transactions contain duplicate transaction IDs"
    )

    return {
        "input_rows": input_rows,
        "classified_rows": metrics_row["classified_rows"],
        "accepted_rows": metrics_row["accepted_rows"],
        "quarantined_rows": metrics_row["quarantined_rows"],
        "duplicate_identical_rows": metrics_row["duplicate_identical_rows"],
        "distinct_accepted_ids": distinct_accepted_ids,
        "rows_with_rescued_data": metrics_row["rows_with_rescued_data"],
    }


reconciliation_metrics = assert_and_summarize_reconciliation(
    bronze,
    classified_transactions,
)

print(f"Bronze input rows:        {reconciliation_metrics['input_rows']:,}")
print(f"Accepted rows:            {reconciliation_metrics['accepted_rows']:,}")
print(f"Quarantined rows:         {reconciliation_metrics['quarantined_rows']:,}")
print(
    "Duplicate identical rows: "
    f"{reconciliation_metrics['duplicate_identical_rows']:,}"
)
print(
    "Distinct accepted IDs:    "
    f"{reconciliation_metrics['distinct_accepted_ids']:,}"
)

# Display batch-level reconciliation summary.
display(
    classified_transactions
    .groupBy("replay_batch", "record_disposition", "duplicate_status")
    .agg(F.count("*").alias("row_count"))
    .orderBy("replay_batch", "record_disposition", "duplicate_status")
)

# Display validation reason frequencies across non-accepted records.
validation_reason_summary = (
    classified_transactions
    .filter(F.size(F.col("validation_reasons")) > 0)
    .select(
        "replay_batch",
        "record_disposition",
        F.explode("validation_reasons").alias("validation_reason")
    )
    .groupBy("replay_batch", "record_disposition", "validation_reason")
    .agg(F.count("*").alias("row_count"))
    .orderBy("replay_batch", "record_disposition", "validation_reason")
)

display(validation_reason_summary)

### Verify classification logic with synthetic fixtures

Exercise `summarize_rescued_data`, `classify_silver_transactions`, and
`assert_and_summarize_reconciliation` on a controlled in-memory fixture
(kept completely separate from Bronze and Silver tables) covering:
- Valid records and whitespace normalization.
- `NULL` and blank required identifiers (`trans_num`, `cc_num`, `merchant`).
- Missing and unparseable event timestamps.
- Missing, malformed, zero, negative, and precision-changing monetary amounts.
- Identical duplicate IDs (both valid and invalid pairs).
- Conflicting duplicate IDs (both all-valid and valid-plus-invalid pairs).
- Multiple missing transaction IDs without accidental row loss.
- Non-empty valid JSON and malformed `_rescued_data` payloads.

In [ ]:
from datetime import date, datetime
from decimal import Decimal
from pyspark.sql import types as T

synthetic_bronze_schema = T.StructType([
    T.StructField("trans_num", T.StringType(), True),
    T.StructField("cc_num", T.StringType(), True),
    T.StructField("merchant", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("amt", T.StringType(), True),
    T.StructField("trans_date_trans_time", T.StringType(), True),
    T.StructField("unix_time", T.StringType(), True),
    T.StructField("source_file", T.StringType(), True),
    T.StructField("source_row_index", T.StringType(), True),
    T.StructField("replay_batch", T.DateType(), True),
    T.StructField("_rescued_data", T.StringType(), True),
    T.StructField("_ingested_file_path", T.StringType(), True),
    T.StructField("_ingestion_timestamp", T.TimestampType(), True),
])


def make_synthetic_row(source_row_index: str, **overrides: object) -> dict:
    """Create a baseline valid synthetic Bronze row with targeted test overrides."""
    base_row = {
        "trans_num": f"TX_{source_row_index}",
        "cc_num": "4000000000000001",
        "merchant": "fraud_Kirlin and Sons",
        "category": "shopping_net",
        "amt": "42.50",
        "trans_date_trans_time": "2019-01-01 00:01:15",
        "unix_time": "1325376075",
        "source_file": "synthetic_fixture.csv",
        "source_row_index": source_row_index,
        "replay_batch": date(2019, 1, 1),
        "_rescued_data": None,
        "_ingested_file_path": "dbfs:/synthetic/part-00000.parquet",
        "_ingestion_timestamp": datetime(2026, 11, 5, 12, 0, 0),
    }
    base_row.update(overrides)
    return base_row


synthetic_rows = [
    # 1-2: Valid baseline and whitespace-padded valid record
    make_synthetic_row("1", trans_num="TX_VALID_1", amt="42.50"),
    make_synthetic_row(
        "2",
        trans_num="  TX_VALID_TRIM  ",
        cc_num=" 4000000000000002 ",
        merchant=" fraud_Sporer-Keebler ",
        amt=" 125.00 ",
        trans_date_trans_time=" 2019-01-01 01:02:03 ",
    ),
    # 3: NULL and blank required identifiers
    make_synthetic_row(
        "3",
        trans_num="TX_NULL_BLANK_REQ",
        cc_num=None,
        merchant="   ",
    ),
    # 4-5: Missing and unparseable event timestamps
    make_synthetic_row(
        "4",
        trans_num="TX_TIME_MISSING",
        trans_date_trans_time="   ",
    ),
    make_synthetic_row(
        "5",
        trans_num="TX_TIME_INVALID",
        trans_date_trans_time="2019-13-45 not-a-timestamp",
    ),
    # 6-11: Missing, malformed, zero, negative, and precision-changing amounts
    make_synthetic_row("6", trans_num="TX_AMT_MISSING", amt=""),
    make_synthetic_row("7", trans_num="TX_AMT_MALFORMED", amt="not_a_number"),
    make_synthetic_row("8", trans_num="TX_AMT_ZERO", amt="0.00"),
    make_synthetic_row("9", trans_num="TX_AMT_NEGATIVE", amt="-15.25"),
    make_synthetic_row("10", trans_num="TX_AMT_PRECISION", amt="19.999"),
    make_synthetic_row("11", trans_num="TX_AMT_NEG_PRECISION", amt="-0.004"),
    # 12-13: Valid identical duplicate IDs (primary retained, extra classified as duplicate_identical)
    make_synthetic_row(
        "12",
        trans_num="TX_DUP_IDENTICAL",
        amt="55.00",
        _ingestion_timestamp=datetime(2026, 11, 5, 12, 0, 0),
    ),
    make_synthetic_row(
        "13",
        trans_num=" TX_DUP_IDENTICAL ",
        amt="55.00",
        _ingestion_timestamp=datetime(2026, 11, 5, 12, 5, 0),
    ),
    # 14-15: Invalid identical duplicate IDs (neither copy may be accepted)
    make_synthetic_row(
        "14",
        trans_num="TX_DUP_IDENTICAL_INVALID",
        amt="-5.00",
    ),
    make_synthetic_row(
        "15",
        trans_num="TX_DUP_IDENTICAL_INVALID",
        amt="-5.00",
    ),
    # 16-17: Conflicting duplicate IDs where both occurrences have valid fields
    make_synthetic_row("16", trans_num="TX_DUP_CONFLICT_VALID", amt="25.00"),
    make_synthetic_row("17", trans_num="TX_DUP_CONFLICT_VALID", amt="99.00"),
    # 18-19: Conflicting duplicate IDs with one valid and one invalid occurrence
    make_synthetic_row("18", trans_num="TX_DUP_CONFLICT_MIXED", amt="40.00"),
    make_synthetic_row("19", trans_num="TX_DUP_CONFLICT_MIXED", amt="corrupted"),
    # 20-21: Multiple missing transaction IDs (must not collapse or be lost)
    make_synthetic_row("20", trans_num=None, amt="10.00"),
    make_synthetic_row("21", trans_num="   ", amt="20.00"),
    # 22-23: Non-empty JSON and malformed rescued data payloads
    make_synthetic_row(
        "22",
        trans_num="TX_RESCUED_JSON",
        _rescued_data='{"_unexpected_col":"extra_value"}',
    ),
    make_synthetic_row(
        "23",
        trans_num="TX_RESCUED_MALFORMED",
        _rescued_data="{malformed_json",
    ),
]

synthetic_bronze = spark.createDataFrame(
    synthetic_rows,
    schema=synthetic_bronze_schema,
)

In [ ]:
# Execute the actual Silver classification and reconciliation functions on synthetic_bronze.
synthetic_classified = classify_silver_transactions(synthetic_bronze)
synthetic_reconciliation = assert_and_summarize_reconciliation(
    synthetic_bronze,
    synthetic_classified,
)

assert synthetic_reconciliation == {
    "input_rows": 23,
    "classified_rows": 23,
    "accepted_rows": 5,
    "quarantined_rows": 17,
    "duplicate_identical_rows": 1,
    "distinct_accepted_ids": 5,
    "rows_with_rescued_data": 2,
}, f"Unexpected synthetic reconciliation counts: {synthetic_reconciliation}"

results_by_index = {
    row["source_row_index"]: row.asDict()
    for row in synthetic_classified.collect()
}

# 1. Valid input and whitespace normalization + raw value preservation
assert results_by_index["1"]["record_disposition"] == "accepted"
assert results_by_index["1"]["duplicate_status"] == "unique_id"
assert results_by_index["1"]["validation_reasons"] == []
assert results_by_index["1"]["amount"] == Decimal("42.50")
assert results_by_index["1"]["event_time"] == datetime(2019, 1, 1, 0, 1, 15)

assert results_by_index["2"]["record_disposition"] == "accepted"
assert results_by_index["2"]["trans_num"] == "TX_VALID_TRIM"
assert results_by_index["2"]["raw_trans_num"] == "  TX_VALID_TRIM  "
assert results_by_index["2"]["cc_num"] == "4000000000000002"
assert results_by_index["2"]["merchant"] == "fraud_Sporer-Keebler"
assert results_by_index["2"]["amount"] == Decimal("125.00")

# 2. NULL and blank required identifiers
assert results_by_index["3"]["record_disposition"] == "quarantined"
assert results_by_index["3"]["cc_num"] is None
assert results_by_index["3"]["merchant"] is None
assert results_by_index["3"]["raw_merchant"] == "   "
assert results_by_index["3"]["validation_reasons"] == [
    "missing_card_id",
    "missing_merchant",
]

# 3. Invalid and missing timestamps
assert results_by_index["4"]["record_disposition"] == "quarantined"
assert results_by_index["4"]["event_time"] is None
assert results_by_index["4"]["validation_reasons"] == ["missing_event_time"]

assert results_by_index["5"]["record_disposition"] == "quarantined"
assert results_by_index["5"]["event_time"] is None
assert (
    results_by_index["5"]["raw_trans_date_trans_time"]
    == "2019-13-45 not-a-timestamp"
)
assert results_by_index["5"]["validation_reasons"] == [
    "unparseable_event_time"
]

# 4. Missing, malformed, non-positive, and precision-changing amounts
assert results_by_index["6"]["record_disposition"] == "quarantined"
assert results_by_index["6"]["validation_reasons"] == ["missing_amount"]

assert results_by_index["7"]["record_disposition"] == "quarantined"
assert results_by_index["7"]["validation_reasons"] == [
    "unparseable_or_out_of_range_amount"
]

assert results_by_index["8"]["record_disposition"] == "quarantined"
assert results_by_index["8"]["validation_reasons"] == ["non_positive_amount"]

assert results_by_index["9"]["record_disposition"] == "quarantined"
assert results_by_index["9"]["validation_reasons"] == ["non_positive_amount"]

assert results_by_index["10"]["record_disposition"] == "quarantined"
assert results_by_index["10"]["amount"] == Decimal("20.00")
assert results_by_index["10"]["validation_reasons"] == [
    "amount_precision_changed"
]

assert results_by_index["11"]["record_disposition"] == "quarantined"
assert results_by_index["11"]["validation_reasons"] == [
    "amount_precision_changed",
    "non_positive_amount",
]

# 5. Identical duplicate transaction IDs (valid pair and invalid pair)
assert results_by_index["12"]["duplicate_status"] == (
    "identical_duplicate_primary"
)
assert results_by_index["12"]["record_disposition"] == "accepted"
assert results_by_index["12"]["validation_reasons"] == []

assert results_by_index["13"]["duplicate_status"] == "identical_duplicate_extra"
assert results_by_index["13"]["record_disposition"] == "duplicate_identical"
assert results_by_index["13"]["validation_reasons"] == [
    "duplicate_identical_transaction_id"
]

assert results_by_index["14"]["duplicate_status"] == (
    "identical_duplicate_primary"
)
assert results_by_index["14"]["record_disposition"] == "quarantined"
assert results_by_index["14"]["validation_reasons"] == ["non_positive_amount"]

assert results_by_index["15"]["duplicate_status"] == "identical_duplicate_extra"
assert results_by_index["15"]["record_disposition"] == "quarantined"
assert results_by_index["15"]["validation_reasons"] == [
    "non_positive_amount",
    "duplicate_identical_transaction_id",
]

# 6. Conflicting duplicate IDs (all-valid pair and valid+invalid pair)
for idx in ("16", "17"):
    assert results_by_index[idx]["duplicate_status"] == "conflicting_duplicate"
    assert results_by_index[idx]["record_disposition"] == "quarantined"
    assert results_by_index[idx]["validation_reasons"] == [
        "conflicting_duplicate_transaction_id"
    ]

assert results_by_index["18"]["duplicate_status"] == "conflicting_duplicate"
assert results_by_index["18"]["record_disposition"] == "quarantined"
assert results_by_index["18"]["validation_reasons"] == [
    "conflicting_duplicate_transaction_id"
]

assert results_by_index["19"]["duplicate_status"] == "conflicting_duplicate"
assert results_by_index["19"]["record_disposition"] == "quarantined"
assert results_by_index["19"]["validation_reasons"] == [
    "unparseable_or_out_of_range_amount",
    "conflicting_duplicate_transaction_id",
]

# 7. Multiple missing transaction IDs preserved without row loss
for idx in ("20", "21"):
    assert results_by_index[idx]["trans_num"] is None
    assert results_by_index[idx]["duplicate_status"] == "missing_id"
    assert results_by_index[idx]["record_disposition"] == "quarantined"
    assert results_by_index[idx]["validation_reasons"] == [
        "missing_transaction_id"
    ]

# 8. Document current behavior for non-empty and malformed _rescued_data
synthetic_rescued_counts, synthetic_rescued_keys = summarize_rescued_data(
    synthetic_bronze
)
rescued_counts_dict = synthetic_rescued_counts.collect()[0].asDict()
assert rescued_counts_dict == {
    "input_rows": 23,
    "rows_with_non_null_rescued_data": 2,
    "rows_with_non_blank_rescued_data": 2,
}

rescued_key_rows = [
    row.asDict() for row in synthetic_rescued_keys.collect()
]
assert {"rescued_keys": ["_unexpected_col"], "row_count": 1} in rescued_key_rows
assert {"rescued_keys": None, "row_count": 1} in rescued_key_rows

# Currently, _rescued_data does not add a validation reason while Bronze
# rescued-data inspection remains pending.
assert results_by_index["22"]["record_disposition"] == "accepted"
assert (
    results_by_index["22"]["_rescued_data"]
    == '{"_unexpected_col":"extra_value"}'
)
assert results_by_index["23"]["record_disposition"] == "accepted"
assert results_by_index["23"]["_rescued_data"] == "{malformed_json"

print("All 23 synthetic fixture rows passed classification and reconciliation checks.")
display(
    synthetic_classified.select(
        "source_row_index",
        "raw_trans_num",
        "trans_num",
        "amount",
        "event_time",
        "duplicate_status",
        "record_disposition",
        "validation_reasons",
        "_rescued_data",
    ).orderBy(F.expr("TRY_CAST(source_row_index AS BIGINT)"))
)

### Rescued-data behavior and unresolved handling

- **Current behavior**:
  - `summarize_rescued_data` counts all non-null and non-blank `_rescued_data` rows. Valid JSON payloads expose their sorted map keys in `rescued_keys`, whereas malformed JSON strings parse to `NULL` (`rescued_keys = null`) under Spark's permissive `from_json` mode.
  - `classify_silver_transactions` preserves `_rescued_data` on every record and counts non-null payloads in `rows_with_rescued_data`, but does **not** yet append a validation reason for non-empty or malformed `_rescued_data`. Otherwise-valid records with populated `_rescued_data` (such as synthetic rows `22` and `23`) are currently classified as `accepted`.
  - `_rescued_data` is excluded from `_business_signature` (treated as ingestion metadata rather than a source business column).
- **Unresolved decision**:
  - Once the Bronze `_rescued_data` inspection output (Cell 4) is confirmed in Databricks, decide whether any non-blank `_rescued_data` (or any rescued key outside an explicitly allowed set, including malformed JSON where `from_json` returns `NULL`) must append an `unexpected_rescued_data` validation reason and route the row to `quarantined`.